In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from src.data_loader import get_con

con = get_con()
print("Connected to DuckDB")

Connected to DuckDB


In [2]:
seller_metrics = con.execute("""
    SELECT
        i.seller_id,
        s.seller_state,
        COUNT(DISTINCT i.order_id)                              AS total_orders,
        SUM(i.price)                                            AS total_revenue,
        AVG(i.price)                                            AS avg_price,
        COUNT(DISTINCT i.product_id)                            AS unique_products,
        AVG(r.review_score)                                     AS avg_review_score,
        COUNT(r.review_id)                                      AS review_count,
        COUNT(CASE WHEN r.review_score >= 4 THEN 1 END) * 100.0 /
            NULLIF(COUNT(r.review_id), 0)                       AS positive_review_pct,
        AVG(DATEDIFF('day',
            o.order_purchase_timestamp::DATE,
            o.order_delivered_customer_date::DATE))             AS avg_delivery_days,
        COUNT(CASE WHEN o.order_delivered_customer_date::DATE
                       <= o.order_estimated_delivery_date::DATE
                   THEN 1 END) * 100.0 / COUNT(*)              AS on_time_pct
    FROM order_items i
    JOIN sellers s  USING (seller_id)
    JOIN orders o   USING (order_id)
    LEFT JOIN reviews r USING (order_id)
    WHERE o.order_status = 'delivered'
    GROUP BY i.seller_id, s.seller_state
    HAVING total_orders >= 20
    ORDER BY total_revenue DESC
""").df()

print(f"Sellers with 20+ orders: {len(seller_metrics):,}")
print(seller_metrics.describe().round(2))

Sellers with 20+ orders: 804
       total_orders  total_revenue  avg_price  unique_products  \
count        804.00         804.00     804.00           804.00   
mean         106.85       13616.68     133.96            31.98   
std          183.07       24441.87     142.05            40.15   
min           20.00         263.13      10.16             1.00   
25%           31.00        3546.62      57.85            11.00   
50%           52.50        6402.46      97.86            20.00   
75%          104.25       12996.81     150.64            36.00   
max         1819.00      226987.93    1633.19           394.00   

       avg_review_score  review_count  positive_review_pct  avg_delivery_days  \
count            804.00        804.00               804.00             804.00   
mean               4.13        119.76                78.04              12.15   
std                0.34        208.62                10.07               3.30   
min                2.27         19.00               

In [3]:
scaler = MinMaxScaler()

# Metrics where higher = better
pos_cols = ["total_revenue", "avg_review_score", 
            "positive_review_pct", "on_time_pct", "unique_products"]

# Metrics where lower = better (we invert)
neg_cols = ["avg_delivery_days"]

df_score = seller_metrics.copy()

# Normalise 0-1
df_score[pos_cols] = scaler.fit_transform(df_score[pos_cols])
df_score[neg_cols] = 1 - scaler.fit_transform(df_score[neg_cols])

# Weighted composite score
weights = {
    "total_revenue":       0.25,
    "avg_review_score":    0.25,
    "positive_review_pct": 0.20,
    "on_time_pct":         0.20,
    "avg_delivery_days":   0.05,
    "unique_products":     0.05,
}

df_score["composite_score"] = sum(
    df_score[col] * w for col, w in weights.items()
)

# Add back original values for display
df_score["orig_revenue"]   = seller_metrics["total_revenue"]
df_score["orig_orders"]    = seller_metrics["total_orders"]
df_score["orig_review"]    = seller_metrics["avg_review_score"]
df_score["orig_on_time"]   = seller_metrics["on_time_pct"]
df_score["seller_state"]   = seller_metrics["seller_state"]
df_score["seller_id_disp"] = seller_metrics["seller_id"].str[:8] + "..."
df_score["rank"]           = df_score["composite_score"].rank(
    ascending=False).astype(int)

top20 = df_score.nlargest(20, "composite_score")
bot20 = df_score.nsmallest(20, "composite_score")

print("TOP 10 SELLERS:")
print(top20[["seller_id_disp","seller_state","orig_orders",
             "orig_revenue","orig_review","orig_on_time",
             "composite_score"]].head(10).to_string(index=False))

TOP 10 SELLERS:
seller_id_disp seller_state  orig_orders  orig_revenue  orig_review  orig_on_time  composite_score
   fa1c13f2...           SP          578     190917.14     4.373913     90.846287         0.798004
   53243585...           BA          348     217940.44     4.128141     97.000000         0.779795
   7e93a43e...           SP          319     165981.49     4.364486     95.341615         0.772395
   4869f7a5...           SP         1124     226987.93     4.139474     89.459930         0.760888
   7a67c85e...           SP         1145     140238.65     4.268462     94.823123         0.722119
   da8622b1...           SP         1311     162303.67     4.075399     93.634628         0.719364
   4a3ca931...           SP         1772     199408.32     3.828630     90.309488         0.716431
   46dc3b2c...           RJ          503     122811.38     4.263566     94.455067         0.707549
   41c2bad7...           RN           20        782.00     4.956522    100.000000         0.6